<a href="https://colab.research.google.com/github/Nawaf-Alorabi/Tw_Customer_Churn_Prediction_System_DeepLearn/blob/main/Mohamed_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/Nawaf-Alorabi/Tw_Customer_Churn_Prediction_System_DeepLearn/blob/main/Mohamed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Telco Customer Churn — Autoencoder for Anomaly Detection

## 1. Problem Definition

**Objective:** Use Autoencoders to detect anomalous customer behavior patterns in the Telco Churn dataset, treating potential churners as anomalies.

**Dataset:** [Telco Customer Churn (IBM / Kaggle)](https://www.kaggle.com/datasets/blastchar/telco-customer-churn) — 7,043 records × 21 features.

**Approach:**
- Data preprocessing with label encoding and MinMax scaling
- **Baseline Autoencoder** — simple encoder-decoder architecture
- **Advanced Denoising Autoencoder** — with L2 regularization, Dropout, Early Stopping, and Gaussian noise injection
- Compare reconstruction error (MSE) and anomaly detection between both models

**Author:** Mohamed

## 2. Imports & Setup

In [3]:
!pip install keras-tuner
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 5.8 MB/s eta 0:00:00


## 3. Data Loading

In [6]:
#from google.colab import files
#uploaded = files.upload()

In [4]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

## 4. Data Preprocessing

### 4.1 Initial Inspection & Cleaning

In [5]:
print(df.isnull().sum())

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


In [6]:
df['TotalCharges'].dropna(inplace=True)

In [7]:
df.isnull().sum()

,0
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


In [8]:
df = df.dropna()

In [9]:
df = df.replace(" ", np.nan)
df = df.apply(pd.to_numeric, errors='coerce')
df = df.fillna(df.median())

### 4.2 Full Preprocessing Pipeline

In [10]:


# Load data
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.columns = df.columns.str.lower()
df = df.drop('customerid', axis=1)
df = df.replace(" ", np.nan)

# Numeric
num_cols = ['tenure','monthlycharges','totalcharges']
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Binary
binary_cols = ['partner','dependents','phoneservice','paperlessbilling']
for col in binary_cols:
    df[col] = df[col].map({'Yes':1,'No':0})
    df['gender'] = df['gender'].map({'Male':1, 'Female':0})

# Categorical
cat_cols = [
    'multiplelines','internetservice','onlinesecurity','onlinebackup',
    'deviceprotection','techsupport','streamingtv','streamingmovies',
    'contract','paymentmethod'
]
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))


# Features only
X = df.drop('churn', axis=1)
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = np.nan_to_num(X_scaled)

X_train, X_test = train_test_split(X_scaled, test_size=0.2, random_state=42)




/usr/local/lib/python3.12/dist-packages/sklearn/utils/_array_api.py:776: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmin(X, axis=axis))
/usr/local/lib/python3.12/dist-packages/sklearn/utils/_array_api.py:793: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmax(X, axis=axis))


In [11]:
print("NaN count:", np.isnan(X_scaled).sum())

print("Inf count:", np.isinf(X_scaled).sum())

X_scaled = np.nan_to_num(X_scaled)

NaN count: 0
Inf count: 0


19


## 5. Model Design

### 5.1 Baseline Autoencoder

In [13]:
input_dim = X_scaled.shape[1]

input_layer = Input(shape=(input_dim,))
encoded = Dense(16, activation='relu')(input_layer)
decoded = Dense(input_dim, activation='sigmoid')(encoded)

baseline_ae = Model(inputs=input_layer, outputs=decoded)
baseline_ae.compile(optimizer='adam', loss='mse')

history_baseline = baseline_ae.fit(
    X_train, X_train,
    validation_data=(X_test,X_test),
    epochs=20,
    batch_size=64,
    verbose=1
)

# Evaluate
X_pred_baseline = baseline_ae.predict(X_test)
mse_baseline = np.mean((X_test - X_pred_baseline)**2, axis=1)
threshold_baseline = np.percentile(mse_baseline, 95)
anomalies_baseline = np.sum(mse_baseline > threshold_baseline)

print("Baseline MSE:", np.mean(mse_baseline))
print("Baseline anomalies:", anomalies_baseline)

Epoch 1/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - loss: 0.1880 - val_loss: 0.1659
Epoch 2/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1473 - val_loss: 0.1321
Epoch 3/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1169 - val_loss: 0.1034
Epoch 4/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0909 - val_loss: 0.0803
Epoch 5/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0717 - val_loss: 0.0643
Epoch 6/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0583 - val_loss: 0.0528
Epoch 7/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0484 - val_loss: 0.0440
Epoch 8/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0407 - val_loss: 0.0372
Epoch 9/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0344 - val_loss: 0.0314
Epoch 10/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0290 - val_loss: 0.0265
Epoch 11/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0245 - val_loss: 0.0223
Epoch 12/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0206 - val_l

## 6. Training & Tuning — Advanced Denoising Autoencoder

### Regularization Techniques Used:
- **L2 Regularization** on encoder/decoder layers
- **Dropout** (0.2) for regularization
- **Early Stopping** (patience=3, restore best weights)
- **Gaussian Noise** injection for denoising

In [15]:
# Add noise
noise_factor = 0.2
X_train_noisy = X_train + noise_factor*np.random.normal(loc=0.0, scale=1.0, size=X_train.shape)
X_train_noisy = np.clip(X_train_noisy, 0., 1.)
X_test_noisy = X_test + noise_factor*np.random.normal(loc=0.0, scale=1.0, size=X_test.shape)
X_test_noisy = np.clip(X_test_noisy, 0., 1.)

input_layer_adv = Input(shape=(input_dim,))
encoded_adv = Dense(32, activation='relu', kernel_regularizer=l2(0.001))(input_layer_adv)
encoded_adv = Dropout(0.2)(encoded_adv)
encoded_adv = Dense(16, activation='relu')(encoded_adv)

decoded_adv = Dense(32, activation='relu', kernel_regularizer=l2(0.001))(encoded_adv)
decoded_adv = Dense(input_dim, activation='sigmoid')(decoded_adv)

advanced_ae = Model(inputs=input_layer_adv, outputs=decoded_adv)
advanced_ae.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history_adv = advanced_ae.fit(
    X_train_noisy, X_train,
    validation_data=(X_test_noisy, X_test),
    epochs=180,
    batch_size=256,
    callbacks=[early_stop],
    verbose=1
)

# Evaluate
X_pred_adv = advanced_ae.predict(X_test_noisy)
mse_adv = np.mean((X_test - X_pred_adv)**2, axis=1)
threshold_adv = np.mean(mse_adv) + 2*np.std(mse_adv)
anomalies_adv = np.sum(mse_adv > threshold_adv)

print("Advanced MSE:", np.mean(mse_adv))
print("Advanced anomalies:", anomalies_adv)
print("Baseline MSE:", np.mean(mse_baseline))
print("Baseline anomalies:", anomalies_baseline)

Epoch 1/180
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 75ms/step - loss: 0.2321 - val_loss: 0.2241
Epoch 2/180
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.2160 - val_loss: 0.2064
Epoch 3/180
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.1970 - val_loss: 0.1859
Epoch 4/180
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.1788 - val_loss: 0.1688
Epoch 5/180
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.1646 - val_loss: 0.1559
Epoch 6/180
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.1536 - val_loss: 0.1451
Epoch 7/180
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.1446 - val_loss: 0.1361
Epoch 8/180
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.1355 - val_loss: 0.1264
Epoch 9/180
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.1277 - val_loss: 0.1188
Epoch 10/180
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.1216 - val_loss: 0.1134
Epoch 11/180
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.1171 - val_loss: 0.1092
Epoch 12/180
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.

## 7. Evaluation

The advanced denoising autoencoder with L2 regularization and dropout provides more robust anomaly detection compared to the baseline, as it learns to reconstruct clean data from noisy inputs, making it more resilient to noise in production data.

## 8. Model Saving & Loading

In [16]:
# Save the advanced autoencoder
advanced_ae.save("telco_churn_autoencoder.h5")
print("Advanced autoencoder saved as telco_churn_autoencoder.h5")

Advanced autoencoder saved as telco_churn_autoencoder.h5


## 9. Inference Pipeline

In [19]:
from tensorflow import keras
import numpy as np

loaded_ae = baseline_ae
threshold_ae = threshold_baseline

def detect_anomaly(sample, model=loaded_ae, threshold=threshold_ae):
    """
    Detect if a customer record is anomalous based on reconstruction error.

    Parameters:
    sample: array-like with shape (n_features,) or (n_samples, n_features)
    model: trained autoencoder
    threshold: MSE threshold for anomaly classification

    Returns:
    dict with reconstruction errors and anomaly flags
    """
    sample = np.array(sample)
    if sample.ndim == 1:
        sample = sample.reshape(1, -1)

    reconstructed = model.predict(sample, verbose=0)
    mse = np.mean((sample - reconstructed)**2, axis=1)
    is_anomaly = (mse > threshold).astype(int)

    return {
        "reconstruction_error": mse,
        "is_anomaly": is_anomaly,
        "threshold": threshold
    }

# Example usage
example = detect_anomaly(X_test[0])
print(example)

{'reconstruction_error': array([0.01922227]), 'is_anomaly': array([1]), 'threshold': np.float64(0.01759057767495677)}
